# 📘 Refined: Consolidate UK Weather Data with Metadata & Logging


- Combine all historical temperature data into a single refined table
- Create a refined dimension table for UK cities
- Create a date table for reporting
- Maintain logs for every operation using a shared logging framework
- All outputs are saved in the `refined` schema

## Library Imports

In [ ]:
# -----------------------------------------------
# 📦 Library Imports
# -----------------------------------------------
import uuid
import time
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import col, lit, to_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType
import sempy.fabric as fabric
import uuid
from datetime import date
from pyspark.sql.functions import col, lit, current_timestamp

## Setup Metadata

In [ ]:
# -----------------------------------------------
# 🏁 Setup Metadata
# -----------------------------------------------
lakehouse_id = fabric.get_lakehouse_id()
workspace_id = fabric.get_workspace_id()
notebook_name = fabric.resolve_item_name(fabric.get_artifact_id())
log_table="log.monitoring_log"
log_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/log/monitoring_log"

## Log Utility Function

In [ ]:
# -----------------------------------------------
# 🧹 Log Utility
# -----------------------------------------------
def log_changes(notebook_name, table_name, log_path, row_before, row_after, row_inserted, user, operation_type, exec_time, message, error_message=""):
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    run_id = str(uuid.uuid4())
    timestamp = datetime.utcnow()
    schema = StructType([
        StructField("notebook_name", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("run_id", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("operation_type", StringType(), True),
        StructField("row_count", IntegerType(), True),
        StructField("message", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("timestamp", TimestampType(), True)
    ])
    log_data = [{
        "notebook_name": notebook_name,
        "table_name": table_name,
        "run_id": run_id,
        "user_name": "prathy.k",
        "operation_type": operation_type,
        "row_count": row_inserted,
        "message": message,
        "error_message": error_message,
        "timestamp": timestamp
    }]
    spark.createDataFrame(log_data, schema=schema).write.format("delta").mode("append").saveAsTable(log_table)
    print(f"✅ Log written for {table_name}")

## Create Refined Weather Data

In [ ]:
# # -----------------------------------------------
# # 🌆 Create Refined Weather Data
# # -----------------------------------------------

# 🔧 Step 0: Configuration
# ------------------------
raw_table_names = [
    "raw.london_hourly_weather",
    "raw.birmingham_hourly_weather",
    "raw.manchester_hourly_weather",
    "raw.newcastle_hourly_weather",
    "raw.leeds_hourly_weather"
]

refined_weather_table = "refined.fact_hourly_weather"  # <-- single source of truth

# 🔧 Step 1: Run metadata
# -----------------------
run_id   = str(uuid.uuid4())
run_date = date.today()

# 🔧 Step 2: Read & union all raw feeds
# -------------------------------------
df_combined = None
for tbl in raw_table_names:
    try:
        df = spark.read.table(tbl)
        df_combined = df if df_combined is None else df_combined.unionByName(df)
        print(f"✅ Loaded {tbl}")
    except Exception as e:
        print(f"⚠️ Skipped {tbl}: {e}")

# 🔧 Step 3: Transform & attach ETL metadata
# -----------------------------------------
df_refined = (
    df_combined
      .withColumn("temperature_2m", col("temperature_2m").cast("double"))
      .select(
         col("date").cast("date").alias("weather_date"),
         "temperature_2m",
         "city",
         "country"
      )
      .withColumn("run_id",    lit(run_id))
      .withColumn("run_date",  lit(run_date))
      .withColumn("ingest_ts", current_timestamp())
)

# 🔧 Step 4: Create refined target if needed
# ------------------------------------------
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {refined_weather_table} (
    weather_date   DATE,
    temperature_2m DOUBLE,
    city           STRING,
    country        STRING,
    run_id         STRING,
    run_date       DATE,
    ingest_ts      TIMESTAMP
  )
  USING DELTA
  PARTITIONED BY (run_date)
""")

# 🔧 Step 5: Append this run’s data
# --------------------------------
df_refined.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(refined_weather_table)

# 🔧 Step 6: Sanity-check
# ----------------------
print(f"Appended {df_refined.count()} rows to {refined_weather_table}")
spark.table(refined_weather_table) \
     .filter(col("run_id") == run_id) \
     .show(truncate=False)

## Create Refined City Dimension

#### Enable VOrdey for Curated Layer - https://learn.microsoft.com/en-us/fabric/data-engineering/delta-optimization-and-v-order?tabs=pyspark

In [ ]:
# %%pyspark
# spark.conf.set('spark.sql.parquet.vorder.default', 'true')

### Resource Profile and make it ready for PBI

https://learn.microsoft.com/en-us/fabric/data-engineering/configure-resource-profile-configurations

In [ ]:
spark.conf.set("spark.fabric.resourceProfile", "readHeavyForPBI")
# We can do this at session level or use a spark environment

In [ ]:
# # -----------------------------------------------
# # 🌆 Create Refined City Dimension
# # -----------------------------------------------

import uuid
from datetime import date
from pyspark.sql.functions import col, lit, current_timestamp

# 1️⃣ Generate run metadata
run_id   = str(uuid.uuid4())
run_date = date.today()        # e.g. 2025-06-12

# 2️⃣ Read raw Delta
raw_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/raw/city_locations"
df_raw = spark.read.format("delta").load(raw_path)

# 3️⃣ Transform & select top 5 UK cities
df_refined = (
    df_raw
      .filter(col("country") == "United Kingdom")
      .withColumn("population_long", col("population").cast("long"))
      .orderBy(col("population_long").desc())
      .limit(5)
      .select(
          "city",
          "lat",
          "lng",
          "country",
          "population_long"
      )
      # 4️⃣ Add our metadata columns
      .withColumn("run_id",    lit(run_id))
      .withColumn("run_date",  lit(run_date))
      .withColumn("ingest_ts", current_timestamp())
)

# 5️⃣ Create refined table if needed
spark.sql("""
  CREATE TABLE IF NOT EXISTS refined.city_locations (
    city            STRING,
    lat             DOUBLE,
    lng             DOUBLE,
    country         STRING,
    population_long LONG,
    run_id          STRING,
    run_date        DATE,
    ingest_ts       TIMESTAMP
  )
  USING DELTA
  PARTITIONED BY (run_date)
""")

# 6️⃣ Write into refined (append)
df_refined.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("refined.city_locations")

# ✅ Verify
print(f"Appended {df_refined.count()} rows to refined.city_locations")
spark.table("refined.city_locations") \
     .filter(col("run_id") == run_id) \
     .show(truncate=False)

## Date Dimension

In [ ]:
from pyspark.sql.functions import (
    sequence, explode, col, year, month, dayofmonth, date_format,
    dayofweek, weekofyear, quarter, to_date, lit, when,
    last_day, concat_ws, lpad
)
from datetime import datetime
import os
spark.sql("DROP TABLE IF EXISTS refined.dim_date")
# Generate date range
start_date = datetime.strptime("2010-01-01", "%Y-%m-%d")
end_date = datetime.today()

# Create base date range DataFrame
df_dates = spark.sql(f"""
SELECT explode(sequence(to_date('{start_date.date()}'), to_date('{end_date.date()}'), interval 1 day)) as date
""")

# Add enriched date attributes
df_dates = (
    df_dates
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("week", weekofyear(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    
    # Names and short forms
    .withColumn("month_name_long", date_format(col("date"), "MMMM"))
    .withColumn("month_name_short", date_format(col("date"), "MMM"))
    .withColumn("week_day_name_long", date_format(col("date"), "EEEE"))
    .withColumn("week_day_name_short", date_format(col("date"), "E"))

    # Sorters
    .withColumn("sort_order", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("month_sort_order", (col("year") * 100 + col("month")).cast("int"))
    .withColumn("week_sort_order", (col("year") * 100 + col("week")).cast("int"))

    # Flags
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), lit(True)).otherwise(lit(False)))
    .withColumn("is_month_start", when(dayofmonth(col("date")) == 1, lit(True)).otherwise(lit(False)))
    .withColumn("is_month_end", when(col("date") == last_day(col("date")), lit(True)).otherwise(lit(False)))
)

# Save to refined table
refined_date_table = "refined.dim_date"
df_dates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("refined.dim_date")

# Log operation
log_changes(
    notebook_name,
    refined_date_table,
    log_path,
    0,
    df_dates.count(),
    df_dates.count(),
    os.getenv("USERNAME"),
    "create",
    0,
    "Full date dimension created with extended metadata"
)